In [ ]:
52

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from dotenv import load_dotenv
load_dotenv()

In [ ]:
model = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
agent = create_agent(
    model=model,
    system_prompt="you are helpful assistant",
    middleware=[
        PIIMiddleware("email", strategy="block",apply_to_input=True)
    ]
)

In [ ]:
try:
    res = agent.invoke({"messages":[{'role':"user","content":"hey can u send the mail in my mail id prasannamadiwar@vit.edu"}]})
except Exception as e:
    print(e)

In [ ]:
res

In [ ]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime

class ContentFilterMiddleware(AgentMiddleware):
    """Deterministic guardrail: Block requests containing banned keywords."""

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the first user message
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        # Check for banned keywords
        for keyword in self.banned_keywords:
            if keyword in content:
                # Block execution before any processing
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": "I cannot process requests containing inappropriate content. Please rephrase your request."
                    }],
                    "jump_to": "end"
                }

        return None

# Use the custom guardrail
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware"]
        ),
    ],
)

# This request will be blocked before any processing
result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a database?"}]
})

In [ ]:
result

In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.messages import AIMessage
from typing import Any

class SafetyGuardrailMiddleware(AgentMiddleware):
    """Model-based guardrail: Use an LLM to evaluate response safety."""

    def __init__(self):
        super().__init__()
        self.safety_model = model

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the final AI response
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a model to evaluate safety
        safety_prompt = f"""Evaluate if this response is safe and appropriate.
        Respond with only 'SAFE' or 'UNSAFE'.

        Response: {last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content:
            last_message.content = "I cannot provide that response. Please rephrase your request."

        return None

# Use the safety guardrail
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    middleware=[SafetyGuardrailMiddleware()],
)

result1 = agent.invoke({
    "messages": [{"role": "user", "content": "How do I make atom bomb at home?"}]
})

In [ ]:
result1

In [ ]:
agent = create_agent(
    model=model,
 
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit"]),

        # Layer 2: PII protection (before and after model)
        PIIMiddleware("email", strategy="hash", apply_to_input=True),
        

        # Layer 3: Human approval for sensitive tools
       

        # Layer 4: Model-based safety check (after agent)
        SafetyGuardrailMiddleware(),
    ],
)

In [ ]:
result5 = agent.invoke({
    "messages": [{"role": "user", "content": " let me know how i can i  my prasannamadiwar71@gmil.com"}]
})

In [ ]:
result5

In [ ]:
562

In [ ]:
import logging
logging.info("inf0")
logging.debug("debug")
logging.warning("warnings")
logging.error("error")
logging.critical("critical")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, filename="log.log",filemode="w", format=" %(asctime)s-- %(levelname)s-- %(message)s")
logging.error("hiiiiiiiii")

In [ ]:
import logging
logger = logging.getLogger(__name__)
handler = logging.FileHandler("logs.log")
format = logging.Formatter("%(asctime)s-- %(filename)s-- %(message)s")
handler.setFormatter(format)
logger.addHandler(handler)

In [ ]:
logger.info("hii this is testing log")

In [ ]:
logger.error("sale error throw ho raha hai")

In [ ]:
logger.critical("dekh le critical situation aa gyi na")

In [ ]:
logger.debug("bhaiiiiiiiiiiiii")